# Race Training

Train a single RL jockey (PPO) against scripted opponents.

**Reward:** delta-progress + finish bonus (1st=10, 2nd=5, 3rd=2) + stamina efficiency bonus (3.0 × stamina used at finish).

**Exhaustion:** Sigmoid-knee degradation (knee=0.20, floor=0.45). Stats degrade gradually with stamina, steep drop below 20%.

**Action space:** 6×9 = 54 discrete actions. Tangential: [-0.25, 0, 0.25, 0.5, 0.75, 1]. Normal: [-1 to 1].

**Observations:** 141 floats. Includes lateral offset (lane position) and stamina drain rate.

**Opponents:** 10 strategies including lane-holding smart pacers (Inside/Outside/Center at ±5m) that the agent must overtake laterally.

**Best-model saving:** The logging callback tracks rolling average reward and saves `best_model.zip` whenever a new peak is reached.

**Auto-resume:** Checkpoints save to Google Drive. If the runtime disconnects, re-run all cells.

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

VERSION = "v18"

TRACK = "tracks/test_oval.json"
HORSE_COUNT = 4
MAX_STEPS = 5000
N_ENVS = 12

TOTAL_TIMESTEPS = 2_000_000

# Google Drive
DRIVE_BASE = "/content/drive/MyDrive/hr-checkpoints"
DRIVE_DIR = f"{DRIVE_BASE}/{VERSION}"

RESUME = 'auto'       # 'auto', False, or path to .zip
LOG_EVERY = 10
SAVE_EVERY = 50_000

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = "/content/hr-simulation"
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/ue-too/hr-simulation.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")
!git log --oneline -3

In [ ]:
!pip install -q uv
!uv pip install --system -e "."

## Resolve Checkpoint

In [ ]:
import glob
from pathlib import Path

os.makedirs(DRIVE_DIR, exist_ok=True)

restore_path = None
restored_timesteps = 0

if RESUME is False:
    print("Starting fresh (RESUME=False)")
elif RESUME == 'auto':
    zips = sorted(glob.glob(os.path.join(DRIVE_DIR, "checkpoint_*.zip")), key=os.path.getmtime)
    if zips:
        restore_path = zips[-1]
        stem = Path(restore_path).stem
        parts = stem.split("_")
        for i, p in enumerate(parts):
            if p == "steps" and i > 0:
                try:
                    restored_timesteps = int(parts[i - 1])
                except ValueError:
                    pass
        print(f"Auto-resume: {restore_path}")
        print(f"  Restored timesteps: {restored_timesteps:,}")
    else:
        final = os.path.join(DRIVE_DIR, "final_model.zip")
        if os.path.exists(final):
            restore_path = final
            restored_timesteps = TOTAL_TIMESTEPS
            print(f"Found final model — training already complete!")
        else:
            print("No checkpoint found — starting fresh")
elif isinstance(RESUME, str):
    restore_path = RESUME
    print(f"Resuming from: {restore_path}")

remaining_timesteps = max(0, TOTAL_TIMESTEPS - restored_timesteps)
print(f"\nVersion: {VERSION}")
print(f"Track: {TRACK} | Horses: {HORSE_COUNT}")
print(f"Total: {TOTAL_TIMESTEPS:,} | Remaining: {remaining_timesteps:,}")
print(f"Checkpoint dir: {DRIVE_DIR}")

## Training

In [ ]:
import time
import json
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv

from horse_racing.env import HorseRacingSingleEnv


class ColabLoggingCallback(BaseCallback):
    """Logs episode stats, saves history, and preserves the best model."""

    def __init__(self, log_every=10, total_timesteps=1_000_000,
                 history_path=None, best_model_path=None, best_window=20):
        super().__init__(verbose=0)
        self._log_every = log_every
        self._total_timesteps = total_timesteps
        self._history_path = history_path
        self._best_model_path = best_model_path
        self._best_window = best_window
        self._best_avg = -float("inf")
        self._ep_count = 0
        self._start = time.time()
        self.history = []

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self._ep_count += 1
                ep = info["episode"]
                self.history.append({
                    "episode": self._ep_count,
                    "timesteps": self.num_timesteps,
                    "reward": float(ep["r"]),
                    "length": int(ep["l"]),
                })
                if self._ep_count % self._log_every == 0:
                    elapsed = time.time() - self._start
                    ts = self.num_timesteps
                    rate = ts / elapsed if elapsed > 0 else 0
                    pct = ts / self._total_timesteps * 100
                    eta_s = (self._total_timesteps - ts) / rate if rate > 0 else 0
                    eta_m = eta_s / 60
                    recent = self.history[-10:]
                    avg_r = np.mean([h["reward"] for h in recent])
                    avg_l = np.mean([h["length"] for h in recent])
                    print(
                        f"  [{pct:5.1f}%] ep {self._ep_count:5d} | "
                        f"reward: {avg_r:8.3f} | "
                        f"len: {avg_l:6.0f} | "
                        f"ts: {ts:>10,} | "
                        f"{rate:,.0f} ts/s | "
                        f"ETA: {eta_m:.0f}m"
                    )

                # Save best model when rolling average improves
                if (self._best_model_path
                        and self._ep_count >= self._best_window):
                    window = [h["reward"] for h in self.history[-self._best_window:]]
                    avg = np.mean(window)
                    if avg > self._best_avg:
                        self._best_avg = avg
                        self.model.save(self._best_model_path)
                        print(f"  ** New best model saved (avg {avg:.2f}, ep {self._ep_count}) **")

                # Save history periodically
                if self._history_path and self._ep_count % (self._log_every * 5) == 0:
                    with open(self._history_path, "w") as f:
                        json.dump(self.history, f)
        return True


def make_env(track_path, horse_count, max_steps):
    def _init():
        return Monitor(HorseRacingSingleEnv(
            track_path=track_path, horse_count=horse_count, max_steps=max_steps,
        ))
    return _init

In [ ]:
if remaining_timesteps <= 0:
    print("Training already complete — skip to evaluation or export.")
else:
    vec_env = SubprocVecEnv([make_env(TRACK, HORSE_COUNT, MAX_STEPS) for _ in range(N_ENVS)])
    policy_kwargs = dict(net_arch=[256, 256])

    # Linear LR decay: 3e-4 → 1e-5 over training
    LR_START = 3e-4
    LR_END = 1e-5

    def linear_schedule(progress_remaining: float) -> float:
        """progress_remaining goes from 1.0 → 0.0 during training."""
        return LR_END + (LR_START - LR_END) * progress_remaining

    if restore_path:
        print(f"Loading model from {restore_path}")
        model = PPO.load(restore_path, env=vec_env, device="cpu")
        model.learning_rate = linear_schedule
    else:
        model = PPO(
            "MlpPolicy", vec_env,
            learning_rate=linear_schedule,
            gamma=0.998,
            gae_lambda=0.95,
            n_steps=2048,
            batch_size=512,
            n_epochs=10,
            clip_range=0.2,
            ent_coef=0.02,
            policy_kwargs=policy_kwargs,
            verbose=0,
            device="cpu",
        )

    print(f"Params: {sum(p.numel() for p in model.policy.parameters()):,}")
    print(f"LR schedule: {LR_START} → {LR_END} (linear decay)")
    print()

In [ ]:
if remaining_timesteps <= 0:
    print("Skipping training.")
else:
    history_path = os.path.join(DRIVE_DIR, "history.json")
    best_model_path = os.path.join(DRIVE_DIR, "best_model")

    checkpoint_cb = CheckpointCallback(
        save_freq=max(SAVE_EVERY // N_ENVS, 1),
        save_path=DRIVE_DIR,
        name_prefix="checkpoint",
    )
    logging_cb = ColabLoggingCallback(
        log_every=LOG_EVERY,
        total_timesteps=remaining_timesteps,
        history_path=history_path,
        best_model_path=best_model_path,
        best_window=20,
    )

    print(f"Training: {remaining_timesteps:,} timesteps")
    print(f"Track: {TRACK} | Horses: {HORSE_COUNT} | Envs: {N_ENVS}")
    print(f"Checkpoints -> {DRIVE_DIR}")
    print(f"Best model  -> {best_model_path}.zip")
    print()

    start_time = time.time()
    try:
        model.learn(
            total_timesteps=remaining_timesteps,
            callback=[checkpoint_cb, logging_cb],
            reset_num_timesteps=False if restore_path else True,
        )
    except KeyboardInterrupt:
        print("\nTraining interrupted.")

    final_path = os.path.join(DRIVE_DIR, "final_model")
    model.save(final_path)
    with open(history_path, "w") as f:
        json.dump(logging_cb.history, f)

    elapsed = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"Done — {elapsed/60:.1f} min, {logging_cb._ep_count} episodes")
    print(f"Final model: {final_path}.zip")
    if logging_cb._best_avg > -float("inf"):
        print(f"Best model:  {best_model_path}.zip (avg {logging_cb._best_avg:.2f})")
    print(f"{'='*60}")
    vec_env.close()

## Training Curves

In [ ]:
import matplotlib.pyplot as plt

history_path = os.path.join(DRIVE_DIR, "history.json")
if 'logging_cb' in dir() and logging_cb.history:
    history = logging_cb.history
elif os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
else:
    history = []
    print("No history found.")

if history:
    rewards = [h["reward"] for h in history]
    lengths = [h["length"] for h in history]

    window = min(50, len(rewards) // 4) or 1
    smooth_r = np.convolve(rewards, np.ones(window)/window, mode='valid')
    smooth_l = np.convolve(lengths, np.ones(window)/window, mode='valid')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(rewards, alpha=0.2, color='blue')
    ax1.plot(range(window-1, len(rewards)), smooth_r, color='blue', linewidth=2)
    ax1.set_xlabel('Episode'); ax1.set_ylabel('Reward'); ax1.set_title('Episode Reward')
    ax1.grid(True, alpha=0.3)
    ax2.plot(lengths, alpha=0.2, color='green')
    ax2.plot(range(window-1, len(lengths)), smooth_l, color='green', linewidth=2)
    ax2.set_xlabel('Episode'); ax2.set_ylabel('Length (ticks)'); ax2.set_title('Episode Length')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"Episodes: {len(rewards)}")
    print(f"Last 10 avg reward: {np.mean(rewards[-10:]):.3f}")
    print(f"Last 10 avg length: {np.mean(lengths[-10:]):.0f}")

## Evaluation

In [ ]:
from horse_racing.action import decode_action

# Prefer best_model (peak performance) over final_model
best_path = os.path.join(DRIVE_DIR, "best_model.zip")
final_path = os.path.join(DRIVE_DIR, "final_model.zip")

if os.path.exists(best_path):
    eval_path = best_path
elif os.path.exists(final_path):
    eval_path = final_path
else:
    zips = sorted(glob.glob(os.path.join(DRIVE_DIR, "checkpoint_*.zip")), key=os.path.getmtime)
    if zips:
        eval_path = zips[-1]
    else:
        raise FileNotFoundError("No model found to evaluate")

eval_model = PPO.load(eval_path, device="cpu")
print(f"Evaluating: {eval_path}\n")

env = HorseRacingSingleEnv(track_path=TRACK, horse_count=HORSE_COUNT, max_steps=MAX_STEPS)
for ep in range(10):
    obs, info = env.reset()
    total_r = 0.0
    steps = 0
    while True:
        action, _ = eval_model.predict(obs, deterministic=True)
        obs, r, term, trunc, info = env.step(int(action))
        total_r += r
        steps += 1
        if term or trunc:
            break
    status = f"#{info['finish_order']}" if info['finish_order'] else "DNF"
    print(f"  ep {ep+1:2d}: reward={total_r:8.3f} | steps={steps:5d} | progress={info['progress']:.3f} | stamina={info['stamina']:.1f} | {status}")
env.close()

## Export to ONNX

In [ ]:
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
from horse_racing.core.observation import OBS_SIZE
from horse_racing.action import NUM_ACTIONS


class PolicyWrapper(nn.Module):
    def __init__(self, policy):
        super().__init__()
        self.features_extractor = policy.features_extractor
        self.mlp_extractor = policy.mlp_extractor
        self.action_net = policy.action_net

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        features = self.features_extractor(obs)
        latent_pi, _ = self.mlp_extractor(features)
        return self.action_net(latent_pi)


# Prefer best_model (peak performance) over final_model
best_path = os.path.join(DRIVE_DIR, "best_model.zip")
final_path = os.path.join(DRIVE_DIR, "final_model.zip")

if os.path.exists(best_path):
    export_path = best_path
elif os.path.exists(final_path):
    export_path = final_path
else:
    zips = sorted(glob.glob(os.path.join(DRIVE_DIR, "checkpoint_*.zip")), key=os.path.getmtime)
    export_path = zips[-1] if zips else None
if not export_path:
    raise FileNotFoundError("No model found to export")

export_model = PPO.load(export_path, device="cpu")
print(f"Exporting from: {export_path}")
wrapper = PolicyWrapper(export_model.policy).cpu()
wrapper.eval()

dummy = torch.zeros(1, OBS_SIZE, dtype=torch.float32)
onnx_path = os.path.join(DRIVE_DIR, f"{VERSION}_jockey.onnx")

torch.onnx.export(
    wrapper, dummy, onnx_path,
    input_names=["obs"], output_names=["actions"],
    dynamic_axes={"obs": {0: "batch"}, "actions": {0: "batch"}},
    opset_version=17,
)

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
session = ort.InferenceSession(onnx_path)
test_obs = np.random.randn(4, OBS_SIZE).astype(np.float32)
with torch.no_grad():
    torch_out = wrapper(torch.from_numpy(test_obs)).numpy()
onnx_out = session.run(["actions"], {"obs": test_obs})[0]
max_diff = np.max(np.abs(torch_out - onnx_out))

print(f"Exported to: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)")
print(f"Max PyTorch vs ONNX diff: {max_diff:.2e}")
assert max_diff < 1e-5
print("Validation passed!")
print(f"\nDrive path: hr-checkpoints/{VERSION}/{VERSION}_jockey.onnx")
print(f"Copy to: apps/horse-racing/public/models/{VERSION}_jockey.onnx")